# graphArt_LM: scoring the per-instance trees (dev + generator test)

`pipeline/run_hier_inst.py`: the **instance adapter** writes the module tree one class
instance at a time (the root from the whole program, every other instance from its class's
code and constructor arguments), code joins the answers; the **hierarchical adapter** then
wires every distinct block instance, and instances whose graphs match share a variant tag.
Scored by `pipeline/score_hier.py` (headline `hier_right`). Dev first (51), then the
generator test (75), each split over both T4s. About 2 h, capped at 2.5 h.

**Before running**: *Add Input* -> both training outputs: `graphart_hier_inst_train`
(folder `qwen2.5-coder-1.5b-graphart-lora-hier-inst`) and `graphreArt_heir_train` (folder
`qwen2.5-coder-1.5b-graphart-lora-hier`), as notebook outputs or uploaded datasets.
*GPU T4 x2*, *Internet on*. Then *Save Version -> Save & Run All (Commit)*.

**When it finishes**, download `graphart_hier_inst_eval.zip`: results, scores, raw replies, logs.

Generated by `scripts/make_kaggle_notebook.py --hier-inst-eval`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   the base both adapters were tuned from (checked against their adapter_config.json).
# TREE_ADAPTER the per-instance adapter (graphart_hier_inst_train.zip: lora/<this name>/).
# WIRE_ADAPTER the hierarchical adapter that wires (graphart_hier_train.zip: lora/<this name>/).
#              Both are found by folder name anywhere under /kaggle/input.
# SPLITS       in order: the 51 dev programs (the criterion, P2a), then the 75 generator test
#              programs; each split is divided between the two GPUs.
# RUN_ARGS     output caps per call; at most 300 instance calls per program.
# TIME_BUDGET_HOURS  the whole session; jobs still running at the end are stopped, and every
#              program finished so far is kept and scored.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; TREE_ADAPTER ="qwen2.5-coder-1.5b-graphart-lora-hier-inst"; WIRE_ADAPTER = "qwen2.5-coder-1.5b-graphart-lora-hier"; SPLITS = ["hier_dev", "hier_gen_test"]; RUN_ARGS = ["--inst-tokens", "2048", "--wire-tokens", "2048", "--max-calls", "300"]; TIME_BUDGET_HOURS = 2.5; PACKAGE_RESERVE_MIN = 6; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "301c1c716edf3ce6c8c3b78666cb2efbf4ef17ead208505b4fc1a82af8094437"
BUNDLE = (
    "..."
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Find the two adapters and the GPUs

In [ ]:
import re, zipfile

INPUT = Path(os.environ.get("GRAPHART_INPUT", "/kaggle/input"))
found = [c.parent for c in INPUT.rglob("adapter_config.json")] if INPUT.exists() else []
ADAPTERS = {}
norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())     # a dataset mounts under its slug: dots become dashes
for name in (TREE_ADAPTER, WIRE_ADAPTER):
    hits = [p for p in found if "checkpoint" not in str(p)
            and norm(name) in {norm(part) for part in p.parts[-4:]}]
    if not hits:
        raise SystemExit(f"no adapter folder {name!r} under {INPUT} (found: {[str(p) for p in found]}) -- "
                         "add both training outputs as inputs (see the top)")
    base = json.loads((hits[0] / "adapter_config.json").read_text()).get("base_model_name_or_path")
    if base != BASE_MODEL:
        raise SystemExit(f"{name} was tuned from {base}, not {BASE_MODEL}")
    ADAPTERS[name] = REPO / "lora" / name
    if not ADAPTERS[name].exists():
        shutil.copytree(hits[0], ADAPTERS[name])
    print(name, "<-", hits[0], flush=True)

n = int(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
                       capture_output=True, text=True).stdout.strip() or 0)
print(f"{n} GPU(s) visible", flush=True)
if n < 1:
    raise SystemExit("no GPU -- set the accelerator to GPU T4 x2")
GPUS = min(n, 2)
LOGS = REPO / "logs"
LOGS.mkdir(exist_ok=True)
ZIP_NAME = "graphart_hier_inst_eval.zip"

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def package():
    with zipfile.ZipFile(WORK / ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", LOGS):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file():
                    z.write(p, p.relative_to(REPO))

## 5. Run: each split divided between the GPUs, dev first

In [ ]:
ENV = dict(os.environ, HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
JOBS = [[] for _ in range(GPUS)]
for split in SPLITS:
    rows = [json.loads(l) for l in open(REPO / f"data/generated/{split}.jsonl", encoding="utf-8")]
    for g in range(GPUS):
        ids = ",".join(r["id"] for k, r in enumerate(rows) if k % GPUS == g)
        JOBS[g].append((f"{split} part {'ab'[g]}",
                        [sys.executable, "-m", "pipeline.run_hier_inst", "--tree-model", str(ADAPTERS[TREE_ADAPTER]),
                         "--wire-model", str(ADAPTERS[WIRE_ADAPTER]), "--split", split, "--ids", ids,
                         "--part", "ab"[g], *RUN_ARGS]))
procs = []
for g, jobs in enumerate(JOBS):
    runner = ("import subprocess, sys\n"
              f"for name, cmd in {jobs!r}:\n"
              "    print(f'##### {name}', flush=True)\n"
              "    subprocess.run(cmd)\n")
    log = open(LOGS / f"gpu{g}.log", "w")
    print(f"GPU {g}: {[name for name, _ in jobs]}", flush=True)
    procs.append((g, log, subprocess.Popen([sys.executable, "-c", runner], stdout=log, stderr=subprocess.STDOUT,
                                           env=dict(ENV, CUDA_VISIBLE_DEVICES=str(g)))))
last = time.time()
while any(p.poll() is None for _, _, p in procs):
    if remaining() < PACKAGE_RESERVE_MIN * 60:
        for g, _, p in procs:
            if p.poll() is None:
                p.kill()
                print(f"GPU {g}: stopped by the time budget -- finished programs are kept", flush=True)
        break
    time.sleep(30)
    if time.time() - last > 600:
        last = time.time()
        for g, _, _ in procs:
            done = [l for l in (LOGS / f"gpu{g}.log").read_text(errors="replace").splitlines()
                    if l.strip().startswith("[") or l.startswith("#####")]
            print(f"  GPU {g}: {done[-1].strip() if done else 'starting'}   ({remaining() / 3600:.1f} h left)",
                  flush=True)
for g, log, p in procs:
    log.close()
    print(f"\n##### GPU {g} (exit {p.returncode})\n" +
          "\n".join((LOGS / f"gpu{g}.log").read_text(errors="replace").splitlines()[-20:]), flush=True)
package()

## 6. Merge the halves, score

In [ ]:
res = REPO / "results"
for parts in {tuple(sorted(res.glob(p.name.split("__part")[0] + "__part*.jsonl")))
              for p in res.glob("*__part*.jsonl")}:
    base = parts[0].name.split("__part")[0]
    split = base.split("__")[-1]
    order = {json.loads(l)["id"]: i for i, l in enumerate(open(REPO / f"data/generated/{split}.jsonl", encoding="utf-8"))}
    recs = sorted((json.loads(l) for p in parts for l in p.open(encoding="utf-8")), key=lambda r: order[r["id"]])
    (res / f"{base}.jsonl").write_text("".join(json.dumps(r) + "\n" for r in recs), encoding="utf-8")
    metas = [json.loads(p.with_name(p.stem + ".meta.json").read_text()) for p in parts]
    meta = dict(metas[0], n=len(recs), of=len(order), failed=sum(m["failed"] for m in metas),
                parts=[p.name for p in parts])
    (res / f"{base}.meta.json").write_text(json.dumps(meta, indent=2))
    print(f"merged {len(parts)} parts -> {base}.jsonl ({len(recs)} of {len(order)} programs)", flush=True)
runs = sorted(str(p.relative_to(REPO)) for p in res.glob("hf__*__hier_inst__*.jsonl") if "__part" not in p.name)
out = subprocess.run([sys.executable, "-m", "pipeline.score_hier", *runs], capture_output=True, text=True)
print(out.stdout[-20000:], out.stderr[-3000:], flush=True)
(LOGS / "scores.txt").write_text(out.stdout)
package()
print(f"done in {(time.time() - START) / 3600:.1f} h -- download {ZIP_NAME}")